# Python Fluency Drills — Data Manipulation & LeetCode

**Purpose:** Fix the specific gap from BofA — knowing what to do but freezing on the mechanics of writing it. Dict iteration, data cleaning, normalisation, comprehensions. Then a curated LeetCode set to continue building pattern recognition.

---

### How to use this notebook
- Every section has a `# YOUR TURN` cell. Implement it yourself first — no autocomplete, no looking it up.
- If you freeze, look at the hint in the markdown above, not the internet.
- After completing each section, close the notebook and do a cold rewrite of the hardest exercise from memory.
- The data is financial — similar to what you'd see at CA-CIB and what a Millennium technical might use.

---

### What's covered
- **Part 1** — Dict mechanics (the thing that froze you)
- **Part 2** — Data cleaning & normalisation
- **Part 3** — List comprehensions & sorting
- **Part 4** — Working with nested structures
- **Part 5** — LeetCode problem set (new problems, ordered by pattern)

In [1]:
# Run this first — no external libraries needed for most drills
from collections import defaultdict, Counter
from typing import List, Dict, Optional
print('Ready.')

Ready.


---
## Part 1 — Dict Mechanics

This is the core gap. You need to be able to write these without thinking.

**The four things to know cold:**
```python
d.items()    # → (key, value) pairs — use when you need both
d.keys()     # → just the keys
d.values()   # → just the values
d.get(k, default)  # → safe access, returns default if key missing
```

**Building a frequency dict — two patterns:**
```python
# Pattern A — manual
freq = {}
for item in items:
    freq[item] = freq.get(item, 0) + 1

# Pattern B — Counter
from collections import Counter
freq = Counter(items)
```
Know both. In interviews, Counter is fine to use — just say what it does.

In [18]:
# 1a — Given a list of currency pairs, count how many times each appears.
# Expected output: {'EURUSD': 3, 'GBPUSD': 2, 'USDJPY': 2, 'EURGBP': 1}
pairs = ['EURUSD', 'GBPUSD', 'USDJPY', 'EURUSD', 'GBPUSD', 'EURUSD', 'EURGBP', 'USDJPY']

# YOUR TURN — do it manually first (no Counter)
freq = {}

for pair in pairs:
    freq[pair] = freq.get(pair, 0) + 1
print(freq)

{'EURUSD': 3, 'GBPUSD': 2, 'USDJPY': 2, 'EURGBP': 1}


In [19]:
# 1b — Now do the same thing with Counter in one line
from collections import Counter 
# YOUR TURN
freq = Counter(pairs)

print(freq)


Counter({'EURUSD': 3, 'GBPUSD': 2, 'USDJPY': 2, 'EURGBP': 1})


In [17]:
# 1c — You have a dict of FX spot rates. Print each pair and its rate.
# Then find the pair with the highest rate.
from collections import Counter 

rates = {
    'EURUSD': 1.0823,
    'GBPUSD': 1.2741,
    'USDJPY': 156.42,
    'USDCHF': 0.9012,
    'AUDUSD': 0.6534
}

for pair, rate in rates.items():
    print(f"{pair}: {rate}")


# YOUR TURN — iterate and print each pair + rate
# Then find the pair with the highest rate (hint: max() with a key)
highest = max(rates, key=rates.get)
print(highest)


EURUSD: 1.0823
GBPUSD: 1.2741
USDJPY: 156.42
USDCHF: 0.9012
AUDUSD: 0.6534
USDJPY


In [21]:
# 1d — Merge two dicts of rates. Where a key appears in both, keep the second dict's value.
# Do it without using {**a, **b} — write the loop manually so you understand what's happening.

rates_morning = {'EURUSD': 1.0800, 'GBPUSD': 1.2700, 'USDJPY': 155.90}
rates_afternoon = {'GBPUSD': 1.2741, 'USDJPY': 156.42, 'AUDUSD': 0.6534}

# YOUR TURN
merged = {}
merged = rates_morning.copy()

for key, value in rates_afternoon.items():
    merged[key] = value

print(merged)





{'EURUSD': 1.08, 'GBPUSD': 1.2741, 'USDJPY': 156.42, 'AUDUSD': 0.6534}


---
## Part 2 — Data Cleaning & Normalisation

This is exactly the kind of thing that comes up in data ops technicals — you're given messy records and need to clean, validate, and reshape them.

**Key patterns:**
```python
# Stripping and lowercasing
s.strip().lower()

# Checking for missing/null-like values
if value is None or value == '' or value == 'N/A':
    ...

# Converting types safely
try:
    val = float(raw)
except (ValueError, TypeError):
    val = None

# Splitting a string into parts
'EURUSD_2024-01-15'.split('_')  # → ['EURUSD', '2024-01-15']
```

In [3]:
# 2a — Clean this list of trade records.
# Rules:
#   - Strip whitespace from all string fields
#   - Lowercase the currency pair
#   - Convert 'amount' to float — if it can't be converted, set to None
#   - Drop any record where 'pair' is missing or empty after stripping

raw_trades = [
    {'pair': '  EURUSD ', 'amount': '1500000', 'side': 'BUY'},
    {'pair': 'GBPUSD',    'amount': 'N/A',     'side': 'sell'},
    {'pair': '',          'amount': '500000',   'side': 'BUY'},
    {'pair': ' USDJPY',   'amount': '2000000',  'side': 'SELL'},
    {'pair': 'AUDUSD ',   'amount': '',         'side': 'BUY'},
    {'pair': ' EURGBP ',  'amount': '750000.5', 'side': 'buy'},
]

# YOUR TURN
cleaned_trades = []
clean_pairs = []
clean_amounts = []

for trades in raw_trades:
    try:
        clean_pairs = trades['pair'].strip(" ").lower()
        if trades['pair'] == '' or trades['pair'] is None:
            continue
        clean_amounts = float(str(trades['amount']))
        cleaned_trades.append((clean_pairs, clean_amounts, trades['side']))
    except ValueError:
        clean_amounts = None
        cleaned_trades.append((clean_pairs, clean_amounts, trades['side']))

print(cleaned_trades)


[('eurusd', 1500000.0, 'BUY'), ('gbpusd', None, 'sell'), ('usdjpy', 2000000.0, 'SELL'), ('audusd', None, 'BUY'), ('eurgbp', 750000.5, 'buy')]


In [9]:
# 2b — Normalise a nested structure into a flat list of records.
# Input: a dict where each key is a date and the value is a list of trades for that date.
# Output: a flat list of dicts, each with 'date', 'pair', 'amount' fields.

trade_log = {
    '2024-01-15': [
        {'pair': 'EURUSD', 'amount': 1000000},
        {'pair': 'GBPUSD', 'amount': 500000},
    ],
    '2024-01-16': [
        {'pair': 'USDJPY', 'amount': 2000000},
    ],
    '2024-01-17': [
        {'pair': 'EURUSD', 'amount': 750000},
        {'pair': 'AUDUSD', 'amount': 300000},
        {'pair': 'USDCHF', 'amount': 900000},
    ],
}

# Expected output:
# [
#   {'date': '2024-01-15', 'pair': 'EURUSD', 'amount': 1000000},
#   {'date': '2024-01-15', 'pair': 'GBPUSD', 'amount': 500000},
#   ... etc
# ]


# YOUR TURN
flat_records = []

# for date, trades in trade_log.items():
#     for trade in trades:
#         flat_records.append(
#             {
#                 'date': date,
#                 'pair': trade['pair'],
#                 'amount': trade['amount']
#             }
#        )
flat_records = [
    {
        'date': date,
        'pair': trade['pair'],
        'amount': trade['amount']
    }
    for date, trades in trade_log.items()
    for trade in trades 
]

print(flat_records)



[{'date': '2024-01-15', 'pair': 'EURUSD', 'amount': 1000000}, {'date': '2024-01-15', 'pair': 'GBPUSD', 'amount': 500000}, {'date': '2024-01-16', 'pair': 'USDJPY', 'amount': 2000000}, {'date': '2024-01-17', 'pair': 'EURUSD', 'amount': 750000}, {'date': '2024-01-17', 'pair': 'AUDUSD', 'amount': 300000}, {'date': '2024-01-17', 'pair': 'USDCHF', 'amount': 900000}]


In [14]:
# 2c — Deduplicate records.
# You have a list of rate snapshots. Some are duplicates (same pair + timestamp).
# Keep the last seen value for each (pair, timestamp) combination.

snapshots = [
    {'pair': 'EURUSD', 'ts': '09:00', 'rate': 1.0800},
    {'pair': 'EURUSD', 'ts': '09:00', 'rate': 1.0805},  # duplicate — keep this one
    {'pair': 'GBPUSD', 'ts': '09:00', 'rate': 1.2700},
    {'pair': 'EURUSD', 'ts': '09:30', 'rate': 1.0812},
    {'pair': 'GBPUSD', 'ts': '09:00', 'rate': 1.2710},  # duplicate — keep this one
    {'pair': 'GBPUSD', 'ts': '09:30', 'rate': 1.2741},
]

# YOUR TURN — output should be a list, not a dict
deduped_dict = {}

for snapshot in snapshots:
    key = (snapshot['pair'], snapshot['ts'])
    deduped_dict[key] = snapshot
deduped = list(deduped_dict.values())
print(deduped)

[{'pair': 'EURUSD', 'ts': '09:00', 'rate': 1.0805}, {'pair': 'GBPUSD', 'ts': '09:00', 'rate': 1.271}, {'pair': 'EURUSD', 'ts': '09:30', 'rate': 1.0812}, {'pair': 'GBPUSD', 'ts': '09:30', 'rate': 1.2741}]


In [18]:
# 2d — Group records by a field.
# Given the flat_records from 2b, group them by 'pair'.
# Output: a dict where keys are pairs and values are lists of records for that pair.

# (use flat_records from 2b — run that cell first)

# YOUR TURN — do it manually first, then try with defaultdict
by_pair = {}
for record in flat_records:
    pair = record['pair']
    if pair not in by_pair:
        by_pair[pair] = []
    by_pair[pair].append(record)
print(f"by_pair: {by_pair}")


by_pair: {'EURUSD': [{'date': '2024-01-15', 'pair': 'EURUSD', 'amount': 1000000}, {'date': '2024-01-17', 'pair': 'EURUSD', 'amount': 750000}], 'GBPUSD': [{'date': '2024-01-15', 'pair': 'GBPUSD', 'amount': 500000}], 'USDJPY': [{'date': '2024-01-16', 'pair': 'USDJPY', 'amount': 2000000}], 'AUDUSD': [{'date': '2024-01-17', 'pair': 'AUDUSD', 'amount': 300000}], 'USDCHF': [{'date': '2024-01-17', 'pair': 'USDCHF', 'amount': 900000}]}


---
## Part 3 — List Comprehensions & Sorting

These are the things you should be able to write in one line without hesitating.

**Patterns to know cold:**
```python
# Basic comprehension
[f(x) for x in items]

# With filter
[f(x) for x in items if condition(x)]

# Dict comprehension
{k: v for k, v in d.items() if condition}

# Sorting by a field
sorted(items, key=lambda x: x['field'])
sorted(items, key=lambda x: x['field'], reverse=True)

# Sorting by multiple fields
sorted(items, key=lambda x: (x['date'], x['amount']))
```

In [ ]:
trades = [
    {'pair': 'EURUSD', 'amount': 1500000, 'side': 'buy',  'date': '2024-01-15'},
    {'pair': 'GBPUSD', 'amount': 500000,  'side': 'sell', 'date': '2024-01-15'},
    {'pair': 'USDJPY', 'amount': 2000000, 'side': 'buy',  'date': '2024-01-16'},
    {'pair': 'EURUSD', 'amount': 750000,  'side': 'sell', 'date': '2024-01-16'},
    {'pair': 'AUDUSD', 'amount': 300000,  'side': 'buy',  'date': '2024-01-17'},
    {'pair': 'EURGBP', 'amount': 1200000, 'side': 'sell', 'date': '2024-01-17'},
]

# 3a — Extract just the pair names as a list (one line)
pairs = None  # YOUR TURN

# 3b — Filter to only 'buy' trades (one line)
buys = None  # YOUR TURN

# 3c — Create a dict mapping pair → amount for all trades over 1,000,000 (one line)
large_trades = None  # YOUR TURN

# 3d — Sort all trades by amount descending
by_size = None  # YOUR TURN

# 3e — Sort by date ascending, then amount descending within each date
by_date_then_size = None  # YOUR TURN

print(pairs)
print(buys)
print(large_trades)
print([t['amount'] for t in by_size])
print([(t['date'], t['amount']) for t in by_date_then_size])

---
## Part 4 — Nested Structures

Real data is always nested. This section drills accessing and reshaping it.

**Safe access patterns:**
```python
# Avoid KeyError
record.get('field', default_value)

# Nested safe access (no library needed)
record.get('meta', {}).get('source', 'unknown')
```

In [ ]:
# 4a — The FOMC voting dataset you built at CA-CIB, simplified.
# Extract a flat list of (member_name, vote) tuples from this nested structure.

fomc_meetings = [
    {
        'date': '2024-01-31',
        'decision': 'hold',
        'votes': [
            {'member': 'Powell',    'vote': 'yes'},
            {'member': 'Jefferson', 'vote': 'yes'},
            {'member': 'Waller',    'vote': 'yes'},
        ]
    },
    {
        'date': '2024-03-20',
        'decision': 'hold',
        'votes': [
            {'member': 'Powell',    'vote': 'yes'},
            {'member': 'Jefferson', 'vote': 'yes'},
            {'member': 'Kashkari', 'vote': 'no'},
        ]
    },
]

# YOUR TURN — produce: [('Powell', 'yes'), ('Jefferson', 'yes'), ... ]
# Include the date too: [('2024-01-31', 'Powell', 'yes'), ...]
all_votes = []


In [ ]:
# 4b — Count how many times each FOMC member voted 'yes' vs 'no' across all meetings.
# Output: {'Powell': {'yes': 2, 'no': 0}, 'Kashkari': {'yes': 0, 'no': 1}, ...}

# YOUR TURN
vote_counts = {}


In [ ]:
# 4c — Some records have missing or malformed nested fields. Handle them gracefully.
# Rules: if 'meta' is missing, use source='unknown'. If 'rate' is missing, skip the record.

feed_records = [
    {'pair': 'EURUSD', 'rate': 1.0823, 'meta': {'source': 'Bloomberg'}},
    {'pair': 'GBPUSD', 'rate': 1.2741, 'meta': {'source': 'Refinitiv'}},
    {'pair': 'USDJPY', 'rate': None,   'meta': {'source': 'Bloomberg'}},  # skip — no rate
    {'pair': 'AUDUSD', 'rate': 0.6534},                                    # meta missing
    {'pair': 'USDCHF', 'rate': 0.9012, 'meta': {}},                       # source missing
]

# YOUR TURN — produce a clean list of {'pair', 'rate', 'source'} dicts
clean_feed = []


---
## Part 5 — LeetCode Problems

These pick up from where the BofA prep left off. Work through them in order.

**For each problem:**
1. Read it twice before touching the keyboard
2. Say the pattern out loud
3. Write your test cases first (use `assert`)
4. Implement, then state time + space complexity

---

### Problem 1 — Group Anagrams (#49) [Medium] — Hashmap

**Why it's here:** Grouping by a computed property is the exact same pattern as grouping trades by pair or FOMC votes by member. Once you see it, you'll see it everywhere.

**Pattern:** For each word, compute a key that's the same for all anagrams (sorted letters). Group words by that key.

```
Input:  ['eat', 'tea', 'tan', 'ate', 'nat', 'bat']
Output: [['eat','tea','ate'], ['tan','nat'], ['bat']]
```
https://neetcode.io/problems/anagram-groups

In [ ]:
def group_anagrams(strs: List[str]) -> List[List[str]]:
    # YOUR TURN
    pass


if __name__ == '__main__':
    # test 1 — basic
    result = group_anagrams(['eat', 'tea', 'tan', 'ate', 'nat', 'bat'])
    assert sorted([sorted(g) for g in result]) == [['ate', 'eat', 'tea'], ['bat'], ['ant', 'nat', 'tan']] or True  # order doesn't matter
    print('Test 1 groups:', [sorted(g) for g in result])

    # test 2 — single word
    assert group_anagrams(['a']) == [['a']]

    # test 3 — empty string
    assert group_anagrams(['']) == [['']]

    print('All tests passed')

# Time complexity:  O(?)
# Space complexity: O(?)

---
### Problem 2 — Top K Frequent Elements (#347) [Medium] — Hashmap + sorting

**Why it's here:** Frequency counting is core to data ops — most active pairs, most frequent data errors, most common vendors. Same pattern.

**Pattern:** Count frequencies, then return the top k by count.

```
Input:  nums=[1,1,1,2,2,3], k=2
Output: [1, 2]
```
https://neetcode.io/problems/top-k-elements-in-list

In [ ]:
def top_k_frequent(nums: List[int], k: int) -> List[int]:
    # YOUR TURN
    # Approach A: Counter + sorted
    # Approach B: Counter + most_common(k)   ← know this exists
    pass


if __name__ == '__main__':
    assert sorted(top_k_frequent([1,1,1,2,2,3], 2)) == [1, 2]
    assert top_k_frequent([1], 1) == [1]
    assert sorted(top_k_frequent([1,2], 2)) == [1, 2]
    print('All tests passed')

# Time complexity:  O(?)
# Space complexity: O(?)

---
### Problem 3 — Valid Palindrome (#125) [Easy] — Two Pointers

**Why it's here:** Gets two-pointer movement into muscle memory. Simple enough to write cold in under 5 minutes — that should be the target.

**Pattern:** Left pointer from start, right from end. Skip non-alphanumeric. Compare chars (lowercase). Move inward.

```
Input:  's = "A man, a plan, a canal: Panama"'
Output: True
```
https://neetcode.io/problems/is-palindrome

In [ ]:
def is_palindrome(s: str) -> bool:
    # YOUR TURN
    # Hint: str.isalnum() tells you if a character is alphanumeric
    pass


if __name__ == '__main__':
    assert is_palindrome('A man, a plan, a canal: Panama') == True
    assert is_palindrome('race a car') == False
    assert is_palindrome(' ') == True   # edge case — only spaces
    assert is_palindrome('a') == True   # single char
    print('All tests passed')

# Time complexity:  O(?)
# Space complexity: O(?)

---
### Problem 4 — Best Time to Buy and Sell Stock (#121) [Easy] — Sliding Window

**Why it's here:** Directly financial. Track minimum seen so far, compute max profit. Clean intro to the sliding window pattern.

**Pattern:** Track min price as you scan. At each step, compute profit if you sold today. Update max profit.

```
Input:  prices = [7,1,5,3,6,4]
Output: 5  (buy at 1, sell at 6)
```
https://neetcode.io/problems/buy-and-sell-crypto

In [ ]:
def max_profit(prices: List[int]) -> int:
    # YOUR TURN
    pass


if __name__ == '__main__':
    assert max_profit([7,1,5,3,6,4]) == 5
    assert max_profit([7,6,4,3,1]) == 0   # prices only fall — no profit possible
    assert max_profit([1]) == 0            # single price
    assert max_profit([2,4,1]) == 2
    print('All tests passed')

# Time complexity:  O(?)
# Space complexity: O(?)

---
### Problem 5 — Longest Substring Without Repeating Characters (#3) [Medium] — Sliding Window + Set

**Why it's here:** Combines sliding window with a set. Slightly harder than #121 — you have to manage when to shrink the window.

**Pattern:** Expand right pointer. If character already in window (use a set), shrink from left until it's not. Track max window size.

```
Input:  'abcabcbb'
Output: 3  ('abc')
```
https://neetcode.io/problems/longest-substring-without-duplicates

In [ ]:
def length_of_longest_substring(s: str) -> int:
    # YOUR TURN
    pass


if __name__ == '__main__':
    assert length_of_longest_substring('abcabcbb') == 3
    assert length_of_longest_substring('bbbbb') == 1
    assert length_of_longest_substring('pwwkew') == 3
    assert length_of_longest_substring('') == 0      # empty string
    assert length_of_longest_substring('au') == 2
    print('All tests passed')

# Time complexity:  O(?)
# Space complexity: O(?)

---
### Problem 6 — 3Sum (#15) [Medium] — Sort + Two Pointers

**Why it's here:** Tests whether you can handle duplicates carefully. Builds on two pointers but requires more careful thinking. This is a real step up in difficulty — don't rush it.

**Pattern:** Sort the array. Fix one element with an outer loop. Use two pointers on the rest. Skip duplicates explicitly.

```
Input:  nums = [-1,0,1,2,-1,-4]
Output: [[-1,-1,2],[-1,0,1]]
```
https://neetcode.io/problems/three-integer-sum

In [ ]:
def three_sum(nums: List[int]) -> List[List[int]]:
    # YOUR TURN
    # Step 1: sort nums
    # Step 2: outer loop fixes nums[i]
    # Step 3: two pointers on nums[i+1:]
    # Step 4: skip duplicates at each level
    pass


if __name__ == '__main__':
    result = three_sum([-1,0,1,2,-1,-4])
    assert sorted([sorted(t) for t in result]) == [[-1,-1,2],[-1,0,1]]

    assert three_sum([0,1,1]) == []
    assert three_sum([0,0,0]) == [[0,0,0]]
    print('All tests passed')

# Time complexity:  O(?)
# Space complexity: O(?)

---
## Cold Rewrite Tracker

After completing a problem, come back here and log it. Be honest.

| Problem | First attempt | Cold rewrite (next day) | Confidence |
|---|---|---|---|
| Group Anagrams (#49) | | | /10 |
| Top K Frequent (#347) | | | /10 |
| Valid Palindrome (#125) | | | /10 |
| Best Time to Buy/Sell (#121) | | | /10 |
| Longest Substring (#3) | | | /10 |
| 3Sum (#15) | | | /10 |

---

**Target before the Millennium technical:** every problem in this notebook at 8+/10 on cold rewrite. The data drills (Parts 1–4) should feel automatic — if you're hesitating on `.items()` or a list comprehension, run those cells again.